# 🧠 การถอดรหัสกล่องขอบเขตสำหรับการตรวจจับวัตถุแบบ Anchor-Based เทียบกับ Anchor-Free

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **การถอดรหัสแบบ Anchor-Based เทียบกับ Anchor-Free**! ในสมุดบันทึกนี้ เราจะ:
1. เขียนฟังก์ชันเพื่อถอดรหัสผลลัพธ์ดิบจากโครงข่ายประสาทเทียมให้อยู่ในรูปพิกัดกล่องขอบเขต ($[x_{\min}, y_{\min}, x_{\max}, y_{\max}]$) ในพื้นที่ของรูปภาพ
2. เปรียบเทียบหลักคณิตศาสตร์ของการถอดรหัสแบบ anchor-based (ซึ่งขึ้นอยู่กับขนาดของกล่องสมอหรือ anchor ล่วงหน้า) และการถอดรหัสแบบ anchor-free (ซึ่งขึ้นอยู่กับระยะออฟเซตจากกึ่งกลางกริด)
3. สร้างอัลกอริทึมการถอดรหัสทั้งสองแบบจากศูนย์โดยใช้ NumPy
4. ถอดรหัสกรณีทดสอบสำหรับเซลล์กริดเฉพาะ ($5,5$) ที่มีขนาดการก้าว (stride) เท่ากับ 32
5. พลอตกล่องขอบเขตที่ถอดรหัสแล้ว ขอบเขตเซลล์กริด และจุดกึ่งกลางเคียงข้างกันโดยใช้ Matplotlib เพื่อแสดงภาพเปรียบเทียบความแตกต่าง

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. การสร้างระบบคณิตศาสตร์ด้วย NumPy

เราสร้างอัลกอริทึมการถอดรหัสทั้งสองรูปแบบตามหลักการพื้นฐาน (first principles)

In [ ]:
def decode_anchor_based(grid_x, grid_y, anchor_w, anchor_h, tx, ty, tw, th, stride=32):
    # 1. Center of bbox in grid coordinates (with sigmoid adjustment)
    bx = 1.0 / (1.0 + np.exp(-tx)) + grid_x
    by = 1.0 / (1.0 + np.exp(-ty)) + grid_y
    
    # 2. Width and height in image space
    bw = anchor_w * np.exp(tw)
    bh = anchor_h * np.exp(th)
    
    # 3. Center scaled to image space
    bx_img = bx * stride
    by_img = by * stride
    
    # 4. Convert to XYXY
    x_min = bx_img - bw / 2
    y_min = by_img - bh / 2
    x_max = bx_img + bw / 2
    y_max = by_img + bh / 2
    return np.array([x_min, y_min, x_max, y_max])

def decode_anchor_free(grid_x, grid_y, l, t, r, b, stride=32):
    # 1. Center point of the cell in image space
    cx = (grid_x + 0.5) * stride
    cy = (grid_y + 0.5) * stride
    
    # 2. Coordinates in image space applying left, top, right, bottom offsets
    x_min = cx - l * stride
    y_min = cy - t * stride
    x_max = cx + r * stride
    y_max = cy + b * stride
    return np.array([x_min, y_min, x_max, y_max])

## 2. การถอดรหัสกรณีทดสอบ (Test Case Decoding)

เราถอดรหัสค่าตัวอย่างสำหรับพิกัดกริด $(5,5)$ ด้วยขนาดการก้าว (stride) เท่ากับ 32 (แมปไปยังพื้นที่พิกัดภาพรอบ ๆ $160$ ถึง $192$)

In [ ]:
stride = 32
grid_x, grid_y = 5.0, 5.0

# Anchor-based inputs
anchor_w, anchor_h = 64.0, 128.0
tx, ty, tw, th = 0.2, -0.4, 0.1, -0.2

box_anchor_based = decode_anchor_based(grid_x, grid_y, anchor_w, anchor_h, tx, ty, tw, th, stride)

# Anchor-free inputs
l, t, r, b = 1.2, 1.8, 1.5, 2.2
box_anchor_free = decode_anchor_free(grid_x, grid_y, l, t, r, b, stride)

print("Decoded Anchor-Based Box:", [round(float(coord), 2) for coord in box_anchor_based])
print("Decoded Anchor-Free Box: ", [round(float(coord), 2) for coord in box_anchor_free])

## 3. การแสดงผลด้วยภาพ (Visualization)

มาแสดงภาพเซลล์กริด $(5,5)$ และกล่องที่ถอดรหัสแล้วในพื้นที่ภาพกัน

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(80, 280)
ax.set_ylim(280, 80) # Inverted Y-axis to match image space

# Draw grid cell (5,5)
cell_x1 = grid_x * stride
cell_y1 = grid_y * stride
rect_cell = patches.Rectangle((cell_x1, cell_y1), stride, stride, linewidth=2, edgecolor='black', facecolor='yellow', alpha=0.1, label='Grid Cell (5,5)')
ax.add_patch(rect_cell)
ax.plot((grid_x + 0.5)*stride, (grid_y + 0.5)*stride, 'ko', label='Cell Center')

# Draw Decoded Anchor-Based Box
w_ab = box_anchor_based[2] - box_anchor_based[0]
h_ab = box_anchor_based[3] - box_anchor_based[1]
rect_ab = patches.Rectangle((box_anchor_based[0], box_anchor_based[1]), w_ab, h_ab, linewidth=3, edgecolor='blue', facecolor='none', label='Decoded Anchor-Based Box')
ax.add_patch(rect_ab)

# Draw Decoded Anchor-Free Box
w_af = box_anchor_free[2] - box_anchor_free[0]
h_af = box_anchor_free[3] - box_anchor_free[1]
rect_af = patches.Rectangle((box_anchor_free[0], box_anchor_free[1]), w_af, h_af, linewidth=3, edgecolor='green', facecolor='none', linestyle='--', label='Decoded Anchor-Free Box')
ax.add_patch(rect_af)

ax.set_title("Anchor-Based vs. Anchor-Free Box Decoding Comparison", fontsize=14)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.show()

## 4. สรุปความแตกต่างที่สำคัญ

| Feature | Anchor-Based (YOLOv5) | Anchor-Free (YOLOv8/11) |
| :--- | :--- | :--- |
| **Box Priors** | รูปทรงของสมอ (anchor shapes) ที่คำนวณไว้ล่วงหน้าด้วย K-Means ($p_w, p_h$) | ไม่มี แบบจำลองระยะห่างของขอบเขตโดยตรง |
| **Target Dimensions** | อัตราส่วนความสูง/ความกว้างแบบลอการิทึม ($e^{t_w}$) | ระยะห่างของพิกเซล/กริดแบบเชิงเส้น ($l, t, r, b$) |
| **Custom Datasets** | จำเป็นต้องคำนวณการจัดกลุ่มใหม่สำหรับรูปทรงที่แปลกตา | ปรับใช้ทั่วไปโดยอัตโนมัติกับรูปทรงใด ๆ ก็ได้ |
| **Output Channels** | สูง (เช่น 3 anchors $\times$ ค่า (5 + num_classes)) | ต่ำ (เช่น ขอบเขต 4 ทิศทาง + ค่า num_classes) |